In [ ]:
import sys
import os
print(os.getcwd())
# adjust this path to your project root
PROJECT_ROOT = os.path.abspath("../src/")  # or "../.." depending on where notebook lives

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Added to path:", PROJECT_ROOT)

In [ ]:
import json
import requests
import pandas as pd
from pathlib import Path

# ---- config ----
BASE_URL = "https://app.pendo.io"          # change if needed
API_KEY = "8ef9fe52-4197-44de-b825-c7edca240a44.us"              # paste key here for quick test
FILE_PATH = Path("src/metadata.txt")        # or the actual path if different

# ---- load payload ----
with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw_payload = json.load(f)

print("Top-level keys:", list(raw_payload.keys()))
print("Request name:", raw_payload["requests"][0]["name"])

# ---- Convert Glen's wrapper to the shape Pendo expects ----
payload = {
    "response": raw_payload.get("response", {"mimeType": "application/json"}),
    "request": raw_payload["requests"][0]
}

# ---- call endpoint ----
url = f"{BASE_URL.rstrip('/')}/api/v1/aggregation"
headers = {
    "Content-Type": "application/json",
    "x-pendo-integration-key": API_KEY
}

resp = requests.post(url, headers=headers, json=payload)

print("Status:", resp.status_code)
print("Content-Type:", resp.headers.get("Content-Type"))
print(resp.text[:3000])  # always inspect before raise_for_status

# show raw response if it fails
resp.raise_for_status()
data = resp.json()

# quick peek
if isinstance(data, dict):
    print("Response keys:", list(data.keys()))
else:
    print("Response type:", type(data))

data

In [ ]:
import pandas as pd

results = data["results"]

df = pd.DataFrame(results)

df["visitorMetadata"] = df["visitorMetadata"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")
df["accountMetadata"] = df["accountMetadata"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

df = df.sort_values("appId").reset_index(drop=True)
df

In [ ]:
rows = []

for r in data["results"]:
    app_id = r["appId"]
    
    for key in r.get("visitorMetadata", []):
        rows.append({
            "appId": app_id,
            "metadataType": "visitor",
            "metadataKey": key
        })
        
    for key in r.get("accountMetadata", []):
        rows.append({
            "appId": app_id,
            "metadataType": "account",
            "metadataKey": key
        })

meta_df = pd.DataFrame(rows).sort_values(["appId", "metadataType", "metadataKey"]).reset_index(drop=True)
meta_df

In [ ]:
summary = (
    meta_df.groupby(["metadataType", "metadataKey"])
    .agg(appCount=("appId", "nunique"))
    .reset_index()
    .sort_values(["metadataType", "appCount", "metadataKey"], ascending=[True, False, True])
)

summary

In [ ]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# ---- wide version (one row per app) ----
wide_df = df.copy()
wide_path = output_dir / "corp_metadata_by_app.csv"
wide_df.to_csv(wide_path, index=False)

# ---- long version (one row per metadata key) ----
long_path = output_dir / "corp_metadata_keys_long.csv"
meta_df.to_csv(long_path, index=False)

# ---- summary version (key frequency across apps) ----
summary_path = output_dir / "corp_metadata_key_summary.csv"
summary.to_csv(summary_path, index=False)

print("Saved files:")
print(wide_path)
print(long_path)
print(summary_path)